# AgentBlame / DCFA: background run on both GPUs

**Start it with Save Version -> Save & Run All (Commit).** It runs on Kaggle's servers, so the laptop can sleep.
It stops itself after `HOURS` and saves everything to the Output tab.

Before saving: Input -> your DCFA dataset **and** (from the 2nd run on) this notebook's own latest Output;
Accelerator -> **GPU T4 x2**; Internet -> **On**.

To continue in the next session: Add Input -> Notebook Output -> this notebook's newest version, then Save & Run All again.
Finished logs are skipped, so each run picks up where the last one stopped.


In [ ]:
%%bash
# SETTINGS: change only the two numbers
mkdir -p /kaggle/temp
echo 30 > /kaggle/temp/setting_SAMPLE     # logs per dataset (same logs in every dataset). Empty = all logs
echo 11 > /kaggle/temp/setting_HOURS      # stop after this many hours (Kaggle stops a run at 12 h; keep a margin)
date +%s > /kaggle/temp/setting_START


In [ ]:
%%bash
# STEP 1: code to /kaggle/temp (Kaggle refuses '&' in file names, so Who&When is uploaded as Who_and_When);
# results, logs and cache go to /kaggle/working, which is saved as the Output
SRC=$(dirname "$(find /kaggle/input -name run.py -path '*DCFA*' -not -path '*/results*' | head -1)")
if [ "$SRC" = "." ]; then echo "ERROR: DCFA dataset not found. Add Input -> your DCFA dataset"; exit 1; fi
CODE=/kaggle/temp/DCFA
rm -rf "$CODE" && mkdir -p "$CODE" && cp -r "$SRC"/. "$CODE"/ && mv "$CODE/Who_and_When" "$CODE/Who&When"
rm -rf "$CODE/.cache"
mkdir -p /kaggle/working/{results,logs,.cache}
# Continue from a previous run of this notebook if its Output is attached as input
echo "Inputs attached:"; ls /kaggle/input
# Any depth: Kaggle mounts a notebook's output several folders down. A finished-results folder holds the model folder.
for d in $(find /kaggle/input -type d -name results -not -path '*/DCFA/*' 2>/dev/null); do
  P=$(dirname "$d"); for x in results logs .cache; do [ -d "$P/$x" ] && cp -rn "$P/$x"/. /kaggle/working/$x/; done
  echo "Restored previous results from $P"
done
for x in results logs .cache; do ln -sfn /kaggle/working/$x "$CODE/$x"; done
echo "Finished logs so far:"; cat /kaggle/working/results/*/*/*.jsonl 2>/dev/null | wc -l
# Fallback: only the zip of a previous run is visible -> unpack it (results and logs; the cache is not in the zip)
if [ -z "$(find /kaggle/input -type d -name results -not -path '*/DCFA/*' 2>/dev/null)" ]; then
  Z=$(find /kaggle/input -name all_results.zip 2>/dev/null | head -1)
  if [ -n "$Z" ]; then
    (cd /kaggle/working && unzip -oq "$Z" 'results/*' 'logs/*') && echo "Restored previous results from $Z"
    echo "Finished logs now:"; cat /kaggle/working/results/*/*/*.jsonl 2>/dev/null | wc -l
  elif [ -n "$(find /kaggle/input -name summary.txt -not -path '*/DCFA/*' 2>/dev/null)" ]; then
    echo "ERROR: a previous run's output is attached but its results were not found. What is attached:"
    find /kaggle/input -maxdepth 5 -not -path '*/DCFA/*' | head -40; exit 1
  else
    echo "(no previous results attached: starting from scratch)"
  fi
fi


In [ ]:
%%bash
# STEP 2: install Ollama and start one server per GPU (ports 11434 and 11435)
set -o pipefail
nvidia-smi --query-gpu=name,memory.total --format=csv || { echo "ERROR: no GPU. Accelerator -> GPU T4 x2"; exit 1; }
curl -sI --max-time 15 https://ollama.com > /dev/null || { echo "ERROR: no internet. Settings -> Internet On"; exit 1; }
apt-get install -y -qq zstd pciutils > /dev/null 2>&1
curl -fsSL https://ollama.com/install.sh | sh > /dev/null || { echo "ERROR: Ollama install failed"; exit 1; }
GPUS=$(nvidia-smi -L | wc -l)
# One server per GPU, started one after another (started together on a fresh machine they can clash and one dies).
# A GPU whose server will not start is left out instead of failing the run; the working ports go to setting_PORTS.
: > /kaggle/temp/setting_PORTS
for g in $(seq 0 $((GPUS - 1))); do
  port=$((11434 + g))
  CUDA_VISIBLE_DEVICES=$g OLLAMA_HOST=127.0.0.1:$port OLLAMA_NUM_PARALLEL=1 OLLAMA_KEEP_ALIVE=-1 \
    setsid nohup ollama serve >> /kaggle/working/logs/ollama_gpu$g.log 2>&1 < /dev/null &
  for i in $(seq 60); do curl -s localhost:$port/api/version > /dev/null && break; sleep 2; done
  if curl -s localhost:$port/api/version > /dev/null; then
    echo "Ollama on GPU $g is up (port $port)"; echo $port >> /kaggle/temp/setting_PORTS
  else
    echo "WARNING: Ollama on GPU $g did not start; continuing without it. Its log:"; tail -20 /kaggle/working/logs/ollama_gpu$g.log
  fi
done
[ -s /kaggle/temp/setting_PORTS ] || { echo "ERROR: no Ollama server started"; exit 1; }
export OLLAMA_HOST=127.0.0.1:$(head -1 /kaggle/temp/setting_PORTS)  # for ollama pull
cd /kaggle/temp/DCFA && set -a && source .env && set +a
ollama pull "$OLLAMA_MODEL" 2>&1 | tail -1 || exit 1
ollama pull "$OLLAMA_EMBED_MODEL" 2>&1 | tail -1 || exit 1
pip install -q python-dotenv tqdm
# Warm-up: load the model at the full 128k context on every server (the first request after loading is slow)
for port in $(cat /kaggle/temp/setting_PORTS); do
  curl -s --max-time 600 localhost:$port/api/chat -d "{\"model\":\"$OLLAMA_MODEL\",\"messages\":[{\"role\":\"user\",\"content\":\"Say ok\"}],\"stream\":false,\"options\":{\"num_ctx\":$OLLAMA_CONTEXT_TOKENS,\"num_predict\":5}}" > /dev/null
  echo "port $port: $(OLLAMA_HOST=127.0.0.1:$port ollama ps 2>&1 | tail -1)"
done
true


In [ ]:
%%bash
# STEP 3: the run. One worker per GPU; they share the job list, so no job runs twice.
cd /kaggle/temp/DCFA
SAMPLE=$(cat /kaggle/temp/setting_SAMPLE); HOURS=$(cat /kaggle/temp/setting_HOURS)
# Stop 15 min before the time limit so there is time to save the Output
export DEADLINE=$(( $(cat /kaggle/temp/setting_START) + HOURS * 3600 - 900 ))
export CLAIMS=/kaggle/temp/claims PY=python SAMPLE
rm -rf "$CLAIMS"
for port in $(cat /kaggle/temp/setting_PORTS); do
  OLLAMA_URL=http://localhost:$port WORKER="port$port " bash run_all.sh > /kaggle/working/logs/worker_$port.log 2>&1 &
done
echo "Workers started: $(wc -l < /kaggle/temp/setting_PORTS)"
wait
echo "== progress"; tail -20 logs/progress.txt


In [ ]:
%%bash
# STEP 4: what is finished, the comparison table, and one zip of everything for download
cd /kaggle/temp/DCFA
echo "== finished logs per dataset / method"
for f in results/*/*/*.jsonl; do printf "%5d  %s\n" "$(grep -vc '\"error\"' "$f")" "${f#results/}"; done
grep -h '"error"' results/*/*/*.jsonl 2>/dev/null | wc -l | xargs echo "failed logs (retried next run):"
python evaluate.py results/* --summary > /kaggle/working/summary.txt 2>&1; head -60 /kaggle/working/summary.txt
cd /kaggle/working && rm -f all_results.zip && zip -rq all_results.zip results logs summary.txt && ls -lh all_results.zip
